In [ ]:
#convert into python lists
import pandas as pd
import numpy as np  

cedar = pd.read_csv("../../processed/final_predicted.csv")

mt_seqs = cedar["mt_seq"].tolist()
wt_seqs = cedar["wt_seq"].tolist()
labels = cedar["label"].values 

epitope_ids = cedar["epitope_id"].astype(str).tolist()

In [ ]:
#device
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
#load module
from transformers import T5Tokenizer
from transformers import T5EncoderModel

model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name)

model = model.to(device)
model.eval() 

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 23543.26it/s]
[transformers] T5EncoderModel LOAD REPORT from: Rostlab/prot_t5_xl_uniref50
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


T5EncoderModel(
  (shared): Embedding(128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=4096, bias=False)
              (k): Linear(in_features=1024, out_features=4096, bias=False)
              (v): Linear(in_features=1024, out_features=4096, bias=False)
              (o): Linear(in_features=4096, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 32)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=1024, out_features=16384, bias=False)
              (wo): Linear(in_features=16384, out_features=1024, bias=False)
              (dropout): Dropo

In [5]:
#format seq for protT5 
#space separated 
def preprocess(seq): 
    return " ".join(list(seq))

In [ ]:
#embedding function

def embed_seq(sequences, batch_size=16): 
    per_residue = []
    eos_embeds = [] #no bos token 

    for i in range(0, len(sequences), batch_size): 
        batch = sequences[i:i+batch_size]

        #reformat to match protT5 
        batch_formatted = [preprocess(s) for s in batch]

        #tokenize  to longest
        inputs = tokenizer(batch_formatted, padding=True, return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

        #run model
        with torch.no_grad(): 
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        #hidden shape
        hidden = outputs.last_hidden_state

        #process each batch by itself #they have different lengths
        for j in range(len(batch)): 
            #unformatted seq
            seq_len = len(batch[j])

            #eos token
            eos = hidden[j, seq_len].cpu().numpy().astype(np.float32)

            #prott5 adds to end only, so residue is first tokens
            per_res = hidden[j, :seq_len].cpu().numpy().astype(np.float32)

            per_residue.append(per_res)
            eos_embeds.append(eos)

    return per_residue, np.array(eos_embeds)

In [ ]:
#run embeddings
mt_per_res, mt_eos = embed_seq(mt_seqs)

In [ ]:
wt_per_res, wt_eos = embed_seq(wt_seqs)

In [ ]:
#sanity checks
print("mt_per_res shape:", len(mt_per_res))
print("wt_per_res shape:", len(wt_per_res))
print("mt_eos shape:", mt_eos.shape)
print("wt_eos shape:", wt_eos.shape)

#also nan checks
print("NaNs in mt_eos:", np.isnan(mt_eos).sum())
print("NaNs in wt_eos:", np.isnan(wt_eos).sum())

mt_mean shape: (6868, 1024)
wt_mean shape: (6868, 1024)
mt_per_res shape: 6868
wt_per_res shape: 6868
NaNs in mt_mean: 0
NaNs in wt_mean: 0


In [ ]:
#save output 
save_dict = {

    #eos token
    "mt_eos": mt_eos,
    "wt_eos": wt_eos,
    
    #metadata
    "labels": labels.astype(np.int32), 
    "lengths": np.array([len(s) for s in mt_seqs], dtype=np.int32), 
    "n_peptides": np.array(len(mt_seqs), dtype=np.int32), 
    "epitope_ids": np.array(epitope_ids, dtype=object)
}

#add per residue
for i in range(len(mt_seqs)): 
    save_dict[f"mt_perres_{i}"] = mt_per_res[i]
    save_dict[f"wt_perres_{i}"] = wt_per_res[i]

np.savez_compressed("prottrans_embeddings.npz", **save_dict)